In [0]:
1. Column Level redaction 
2. Row Level Security
3. Data Masking

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select customer_id, customer_name, date_of_birth, email,member_since, telephone from dev.naval_silver.customers_cleaned

In [0]:
grant use catalog on catalog dev to `account users`;
grant use schema on schema dev.naval_silver to `account users`;
grant select on table dev.naval_silver.customers_detials to `account users`

In [0]:
select * from dev.naval_silver.customers_detials

Column Level

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select 
customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('dataeng') THEN 'REDACTED'
    ELSE email
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select 
customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('dataeng') THEN email
    ELSE 'REDACTED'
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

In [0]:
create or replace view dev.naval_silver.customers_detials as 
select 
CASE WHEN
    is_account_group_member('dataeng') THEN customer_id
    ELSE 'REDACTED'
  END AS customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('dataeng') THEN email
    ELSE 'REDACTED'
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

Row Level

In [0]:
select * from dev.naval_silver.order_cleaned

In [0]:
grant use catalog on catalog dev to `account users`;
grant use schema on schema dev.naval_silver to `account users`;
grant select on table dev.naval_silver.order_transfer to `account users`

In [0]:
create or replace  view dev.naval_silver.order_transfer as 
select * from dev.naval_silver.order_cleaned 
where CASE
    WHEN is_account_group_member('account users') THEN payment_method ='Bank Transfer'
    ELSE True
  END;

In [0]:
-- Step 1: Create a mapping table for group -> allowed payment methods
CREATE OR REPLACE TABLE dev.naval_silver.access_control (
  group_name STRING,
  allowed_payment_method STRING
);

-- Insert allowed values per group
INSERT INTO dev.naval_silver.access_control VALUES
  ('account users', 'Bank Transfer'),
  ('account users', 'Credit Card'),
  ('dataeng', 'Bank Transfer'),
  ('dataeng', 'Credit Card'),
  ('dataeng', 'PayPal'),
  ('dataeng', 'Debit Card');

In [0]:
-- Step 2: Create view that filters dynamically based on the mapping table
CREATE OR REPLACE VIEW dev.naval_silver.order_transfer AS
SELECT * FROM dev.naval_silver.order_cleaned
WHERE payment_method IN (
  SELECT allowed_payment_method
  FROM dev.naval_silver.access_control ac
  WHERE is_account_group_member(ac.group_name)
);

In [0]:
select * from dev.naval_silver.order_transfer

Data Masking 

In [0]:
CREATE OR REPLACE FUNCTION dev.naval_silver.datamask(x STRING)
  RETURNS STRING
  RETURN CONCAT(REPEAT("*", LENGTH(x) - 2), RIGHT(x, 2)
); 

In [0]:
create or replace view dev.naval_silver.customers_detials_mask as 
select 
customer_id, 
customer_name, 
date_of_birth, 
CASE WHEN
    is_account_group_member('business') THEN dev.naval_silver.datamask(email)
    ELSE email
  END AS email,
member_since, 
telephone 
from dev.naval_silver.customers_cleaned

In [0]:
select * from dev.naval_silver.customers_detials_mask